# Adding, Updating and Dropping

**Chapter 06 | pandas | 27_pandas / theory**

Raw data is never in the shape a model or a report needs. In this chapter we learn to **reshape a table**: create new columns, change values, rename, and remove what we do not need.
In ML, creating new columns from old ones is called **feature engineering**, and it often improves a model more than changing the algorithm.

**Contents**
1. The toolbox
2. Our dataset
3. Adding columns
4. Updating values
5. Changing data types
6. Renaming
7. Dropping columns and rows
8. Adding rows
9. Moving and reordering columns
10. Method chaining: `assign` and `pipe`
11. Mini AI/ML example: feature engineering
12. Pandas 3.x notes
13. Common mistakes
14. Interview questions
15. Practice questions with solutions
16. Cheat sheet

In [2]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. The toolbox

| Task | Tool |
|---|---|
| Add / overwrite a column | `df["new"] = ...` |
| Add a column inside a chain | `df.assign(new=...)` |
| Add a column at a given position | `df.insert(pos, "new", values)` |
| Two-way / many-way condition | `np.where`, `np.select` |
| Bin numbers into groups | `pd.cut`, `pd.qcut` |
| Replace values | `replace`, `map`, `loc` |
| Change dtype | `astype`, `pd.to_numeric`, `pd.to_datetime` |
| Rename | `rename`, `df.columns = [...]`, `add_prefix` |
| Drop | `drop(columns=...)`, `drop(index=...)`, `pop`, `del` |
| Add rows | `pd.concat`, `df.loc[new_label] = ...` |
| Reorder columns | `df[[...]]`, `insert`, `reindex` |
| Text to numbers for ML | `map`, `pd.get_dummies` |

**Golden rule of this chapter:** most pandas methods **return a new DataFrame**. The original changes only if you assign the result back (`df = df.drop(...)`).

---
# 2. Our dataset

In [3]:
df = pd.DataFrame({
    "name":       ["Ravi", "Asha", "Karan", "Neha", "Amit", "Sara", "Dev", "Pooja"],
    "city":       ["Indore", "Bhopal", "Delhi", "Indore", "Pune", "Bhopal", "Delhi", "Indore"],
    "hours":      [5.5, 7.0, 3.0, 6.5, 4.0, 8.0, 2.5, 6.0],
    "attendance": [80, 90, 65, 85, 70, 95, 60, 88],
    "prev_score": [70, 78, 55, 72, 60, 85, 50, 68],
    "score":      [85, 92, 58, 88, 64, 97, 45, 81],
    "internet":   ["Yes", "Yes", "No", "Yes", "No", "Yes", "No", "Yes"],
})
df

,name,city,hours,attendance,prev_score,score,internet
0,Ravi,Indore,5.5,80,70,85,Yes
1,Asha,Bhopal,7.0,90,78,92,Yes
2,Karan,Delhi,3.0,65,55,58,No
3,Neha,Indore,6.5,85,72,88,Yes
4,Amit,Pune,4.0,70,60,64,No
5,Sara,Bhopal,8.0,95,85,97,Yes
6,Dev,Delhi,2.5,60,50,45,No
7,Pooja,Indore,6.0,88,68,81,Yes


---
# 3. Adding columns

## 3.1 A constant value (same for every row)

In [4]:
d = df.copy()
d["country"] = "India"
d["passed_before"] = False
d[["name", "country", "passed_before"]].head(3)

,name,country,passed_before
0,Ravi,India,False
1,Asha,India,False
2,Karan,India,False


## 3.2 From other columns (vectorized maths)

No loops. The operation runs on whole columns at once.

In [5]:
d = df.copy()

d["improvement"] = d["score"] - d["prev_score"]
d["score_per_hour"] = (d["score"] / d["hours"]).round(2)
d["effort_index"] = (d["hours"] * d["attendance"] / 100).round(2)

d[["name", "score", "prev_score", "improvement", "score_per_hour", "effort_index"]]

,name,score,prev_score,improvement,score_per_hour,effort_index
0,Ravi,85,70,15,15.45,4.40
1,Asha,92,78,14,13.14,6.30
2,Karan,58,55,3,19.33,1.95
3,Neha,88,72,16,13.54,5.52
4,Amit,64,60,4,16.00,2.80
5,Sara,97,85,12,12.12,7.60
6,Dev,45,50,-5,18.00,1.50
7,Pooja,81,68,13,13.50,5.28


## 3.3 From a condition: `np.where` and `np.select`

In [6]:
d = df.copy()

# two outcomes
d["result"] = np.where(d["score"] >= 60, "Pass", "Fail")

# many outcomes
conditions = [d["score"] >= 90, d["score"] >= 75, d["score"] >= 60]
labels = ["A", "B", "C"]
d["grade"] = np.select(conditions, labels, default="D")

d[["name", "score", "result", "grade"]]

,name,score,result,grade
0,Ravi,85,Pass,B
1,Asha,92,Pass,A
2,Karan,58,Fail,D
3,Neha,88,Pass,B
4,Amit,64,Pass,C
5,Sara,97,Pass,A
6,Dev,45,Fail,D
7,Pooja,81,Pass,B


## 3.4 Binning numbers into groups: `pd.cut` and `pd.qcut`

- `pd.cut`: you decide the **bin edges**
- `pd.qcut`: pandas makes bins with an **equal number of rows** (quantiles)

In [7]:
d = df.copy()

d["score_band"] = pd.cut(d["score"], bins=[0, 60, 75, 90, 100], labels=["Low", "Mid", "High", "Top"])
d["hours_level"] = pd.qcut(d["hours"], q=3, labels=["Light", "Medium", "Heavy"])

print(d[["name", "score", "score_band", "hours", "hours_level"]])
print()
print(d["score_band"].value_counts())

    name  score score_band  hours hours_level
0   Ravi     85       High    5.5      Medium
1   Asha     92        Top    7.0       Heavy
2  Karan     58        Low    3.0       Light
3   Neha     88       High    6.5       Heavy
4   Amit     64        Mid    4.0       Light
5   Sara     97        Top    8.0       Heavy
6    Dev     45        Low    2.5       Light
7  Pooja     81       High    6.0      Medium

score_band
High    3
Low     2
Top     2
Mid     1
Name: count, dtype: int64


## 3.5 From text columns (`.str`) and mapping with a dict

In [8]:
d = df.copy()

d["name_upper"] = d["name"].str.upper()
d["initial"] = d["name"].str[0]
d["name_len"] = d["name"].str.len()

# map(): translate values using a dictionary
d["internet_flag"] = d["internet"].map({"Yes": 1, "No": 0})
d["tier"] = d["city"].map({"Indore": "Tier-2", "Bhopal": "Tier-2", "Pune": "Tier-1", "Delhi": "Tier-1"})

d[["name", "name_upper", "initial", "name_len", "internet", "internet_flag", "city", "tier"]]

,name,name_upper,initial,name_len,internet,internet_flag,city,tier
0,Ravi,RAVI,R,4,Yes,1,Indore,Tier-2
1,Asha,ASHA,A,4,Yes,1,Bhopal,Tier-2
2,Karan,KARAN,K,5,No,0,Delhi,Tier-1
3,Neha,NEHA,N,4,Yes,1,Indore,Tier-2
4,Amit,AMIT,A,4,No,0,Pune,Tier-1
5,Sara,SARA,S,4,Yes,1,Bhopal,Tier-2
6,Dev,DEV,D,3,No,0,Delhi,Tier-1
7,Pooja,POOJA,P,5,Yes,1,Indore,Tier-2


> `map` with a dict gives **NaN** for any value that is not a key. That is useful for catching unexpected values.

## 3.6 From a list, array or another Series

- A **list/array** must have **exactly the same length** as the table (assigned by position).
- A **Series** is matched by **index label** (alignment), like in chapter 02.

In [9]:
d = df.copy()

d["roll_no"] = [1, 2, 3, 4, 5, 6, 7, 8]       # length 8 = OK
d[["name", "roll_no"]].head(3)

,name,roll_no
0,Ravi,1
1,Asha,2
2,Karan,3


In [10]:
# wrong length -> error
try:
    d["bad"] = [1, 2, 3]
except ValueError as e:
    print("ValueError:", e)

ValueError: Length of values (3) does not match length of index (8)


In [11]:
# Series is aligned by index label, missing labels become NaN
extra = pd.Series({0: "A", 1: "B", 5: "C"})
d["batch"] = extra
d[["name", "batch"]]

,name,batch
0,Ravi,A
1,Asha,B
2,Karan,NaN
3,Neha,NaN
4,Amit,NaN
5,Sara,C
6,Dev,NaN
7,Pooja,NaN


## 3.7 `assign()`: add columns and get a new DataFrame back

Useful in method chains. Can use `lambda` to refer to the **current** state of the DataFrame.

In [12]:
new_df = df.assign(
    improvement=lambda x: x["score"] - x["prev_score"],
    passed=lambda x: x["score"] >= 60,
)
new_df[["name", "score", "improvement", "passed"]].head(4)

,name,score,improvement,passed
0,Ravi,85,15,True
1,Asha,92,14,True
2,Karan,58,3,False
3,Neha,88,16,True


In [13]:
print("original columns:", df.columns.tolist())     # df itself is unchanged

original columns: ['name', 'city', 'hours', 'attendance', 'prev_score', 'score', 'internet']


## 3.8 `insert()`: add a column at a chosen position

`df["new"] = ...` always adds at the **end**. `insert(position, name, values)` puts it where you want.

In [14]:
d = df.copy()
d.insert(1, "roll_no", range(1, len(d) + 1))
d.head(3)

,name,roll_no,city,hours,attendance,prev_score,score,internet
0,Ravi,1,Indore,5.5,80,70,85,Yes
1,Asha,2,Bhopal,7.0,90,78,92,Yes
2,Karan,3,Delhi,3.0,65,55,58,No


## 3.9 Preview: a column from group statistics

`transform` gives every row the statistic of its **group** (full groupby comes in chapter 10). Very useful for features such as "difference from city average".

In [15]:
d = df.copy()
d["city_avg_score"] = d.groupby("city")["score"].transform("mean").round(1)
d["vs_city_avg"] = (d["score"] - d["city_avg_score"]).round(1)
d[["name", "city", "score", "city_avg_score", "vs_city_avg"]]

,name,city,score,city_avg_score,vs_city_avg
0,Ravi,Indore,85,84.7,0.3
1,Asha,Bhopal,92,94.5,-2.5
2,Karan,Delhi,58,51.5,6.5
3,Neha,Indore,88,84.7,3.3
4,Amit,Pune,64,64.0,0.0
5,Sara,Bhopal,97,94.5,2.5
6,Dev,Delhi,45,51.5,-6.5
7,Pooja,Indore,81,84.7,-3.7


---
# 4. Updating values

## 4.1 Replace a whole column

In [16]:
d = df.copy()
d["hours"] = d["hours"] * 60          # hours -> minutes
d["hours"].head(3)

0    330.0
1    420.0
2    180.0
Name: hours, dtype: float64

## 4.2 Update only some rows: `loc[condition, column] = value`

In [17]:
d = df.copy()

d.loc[d["score"] < 50, "score"] = 50                         # set a minimum
d.loc[d["city"] == "Delhi", "city"] = "New Delhi"            # fix one value
d.loc[d["hours"] > 7, ["attendance", "internet"]] = [100, "Yes"]    # several columns

d

,name,city,hours,attendance,prev_score,score,internet
0,Ravi,Indore,5.5,80,70,85,Yes
1,Asha,Bhopal,7.0,90,78,92,Yes
2,Karan,New Delhi,3.0,65,55,58,No
3,Neha,Indore,6.5,85,72,88,Yes
4,Amit,Pune,4.0,70,60,64,No
5,Sara,Bhopal,8.0,100,85,97,Yes
6,Dev,New Delhi,2.5,60,50,50,No
7,Pooja,Indore,6.0,88,68,81,Yes


## 4.3 `replace()`: swap specific values

In [18]:
d = df.copy()
d["internet"] = d["internet"].replace({"Yes": "Y", "No": "N"})
d["city"] = d["city"].replace("Bhopal", "Bhopal City")
d[["name", "city", "internet"]].head(4)

,name,city,internet
0,Ravi,Indore,Y
1,Asha,Bhopal City,Y
2,Karan,Delhi,N
3,Neha,Indore,Y


## 4.4 `clip()`, `round()`, and other column methods

In [19]:
d = df.copy()
d["score"] = d["score"].clip(lower=50, upper=95)       # limit values to a range
d["hours"] = d["hours"].round(0)
d[["name", "score", "hours"]]

,name,score,hours
0,Ravi,85,6.0
1,Asha,92,7.0
2,Karan,58,3.0
3,Neha,88,6.0
4,Amit,64,4.0
5,Sara,95,8.0
6,Dev,50,2.0
7,Pooja,81,6.0


## 4.5 `map()` and `apply()` with a function

Use them **only when no vectorized method exists**. Vectorized is much faster.

In [20]:
d = df.copy()

def feedback(score):
    if score >= 90:
        return "Excellent"
    elif score >= 60:
        return "Good"
    return "Needs work"

d["feedback"] = d["score"].apply(feedback)
d[["name", "score", "feedback"]]

,name,score,feedback
0,Ravi,85,Good
1,Asha,92,Excellent
2,Karan,58,Needs work
3,Neha,88,Good
4,Amit,64,Good
5,Sara,97,Excellent
6,Dev,45,Needs work
7,Pooja,81,Good


In [21]:
# Row-wise apply (axis=1): the function receives a whole row. Slow, avoid on big data.
d["summary"] = d.apply(lambda r: f"{r['name']} from {r['city']} scored {r['score']}", axis=1)
d["summary"].head(3).tolist()

['Ravi from Indore scored 85',
 'Asha from Bhopal scored 92',
 'Karan from Delhi scored 58']

Speed order (best to worst): **vectorized operation > `np.where` / `np.select` > `map` with dict > `apply` on a column > `apply(axis=1)`**.

---
# 5. Changing data types

In [22]:
raw = pd.DataFrame({
    "price": ["10", "20", "abc", "40"],
    "qty": [1, 2, 3, 4],
    "date": ["2026-01-05", "2026-02-10", "2026-03-15", "2026-04-20"],
    "city": ["Indore", "Delhi", "Indore", "Pune"],
})
print(raw.dtypes)

price      str
qty      int64
date       str
city       str
dtype: object


In [23]:
raw["qty"] = raw["qty"].astype("float64")                          # int -> float
raw["price"] = pd.to_numeric(raw["price"], errors="coerce")        # bad text -> NaN
raw["date"] = pd.to_datetime(raw["date"])                          # text -> datetime
raw["city"] = raw["city"].astype("category")                       # saves memory, good for ML

print(raw.dtypes)
raw

price           float64
qty             float64
date     datetime64[us]
city           category
dtype: object


,price,qty,date,city
0,10.0,1.0,2026-01-05,Indore
1,20.0,2.0,2026-02-10,Delhi
2,NaN,3.0,2026-03-15,Indore
3,40.0,4.0,2026-04-20,Pune


- `astype("int64")` fails if the column has NaN (plain int cannot hold NaN). Use `"Int64"` (capital I, nullable) or fill first.
- `errors="coerce"` turns unparseable values into NaN instead of raising an error. Very useful on dirty data.
- Datetime gives access to `.dt.year`, `.dt.month`, `.dt.day_name()` (chapter 13).

In [24]:
raw["month"] = raw["date"].dt.month
raw["day_name"] = raw["date"].dt.day_name()
raw[["date", "month", "day_name"]]

,date,month,day_name
0,2026-01-05,1,Monday
1,2026-02-10,2,Tuesday
2,2026-03-15,3,Sunday
3,2026-04-20,4,Monday


---
# 6. Renaming

## 6.1 `rename()` for chosen columns

In [25]:
d = df.rename(columns={"hours": "study_hours", "prev_score": "previous_score"})
d.columns.tolist()

['name',
 'city',
 'study_hours',
 'attendance',
 'previous_score',
 'score',
 'internet']

## 6.2 Rename all columns at once

In [26]:
d = df.copy()
d.columns = ["Name", "City", "Hours", "Attendance", "Prev Score", "Score", "Internet"]   # list must match length
print(d.columns.tolist())

['Name', 'City', 'Hours', 'Attendance', 'Prev Score', 'Score', 'Internet']


In [27]:
# Clean messy names: lower case, no spaces -> snake_case (use this on every new dataset)
d.columns = d.columns.str.strip().str.lower().str.replace(" ", "_")
print(d.columns.tolist())

['name', 'city', 'hours', 'attendance', 'prev_score', 'score', 'internet']


In [28]:
# rename can take a function
print(df.rename(columns=str.upper).columns.tolist())
print(df.add_prefix("f_").columns.tolist())
print(df.add_suffix("_x").columns.tolist())

['NAME', 'CITY', 'HOURS', 'ATTENDANCE', 'PREV_SCORE', 'SCORE', 'INTERNET']
['f_name', 'f_city', 'f_hours', 'f_attendance', 'f_prev_score', 'f_score', 'f_internet']
['name_x', 'city_x', 'hours_x', 'attendance_x', 'prev_score_x', 'score_x', 'internet_x']


In [29]:
# Rename index labels
df.head(3).rename(index={0: "first", 1: "second", 2: "third"})

,name,city,hours,attendance,prev_score,score,internet
first,Ravi,Indore,5.5,80,70,85,Yes
second,Asha,Bhopal,7.0,90,78,92,Yes
third,Karan,Delhi,3.0,65,55,58,No


---
# 7. Dropping columns and rows

## 7.1 `drop()`: always say `columns=` or `index=` clearly

In [30]:
d = df.drop(columns=["internet", "prev_score"])
d.columns.tolist()

['name', 'city', 'hours', 'attendance', 'score']

In [31]:
d = df.drop(index=[0, 2])               # drop rows by index label
d[["name"]]

,name
1,Asha
3,Neha
4,Amit
5,Sara
6,Dev
7,Pooja


In [32]:
# Drop rows by a condition = keep the opposite
d = df[df["score"] >= 60]
len(df), "->", len(d)

(8, '->', 6)

In [33]:
# A label that does not exist raises KeyError. errors="ignore" skips it
try:
    df.drop(columns=["not_there"])
except KeyError as e:
    print("KeyError:", e)

print(df.drop(columns=["not_there", "internet"], errors="ignore").columns.tolist())

KeyError: "['not_there'] not found in axis"
['name', 'city', 'hours', 'attendance', 'prev_score', 'score']


## 7.2 `pop()` and `del`

In [34]:
d = df.copy()
removed = d.pop("internet")        # removes the column AND returns it as a Series
print(removed.head(3).tolist())
print(d.columns.tolist())

['Yes', 'Yes', 'No']
['name', 'city', 'hours', 'attendance', 'prev_score', 'score']


In [35]:
d = df.copy()
del d["city"]                      # removes the column, no return value
print(d.columns.tolist())

['name', 'hours', 'attendance', 'prev_score', 'score', 'internet']


## 7.3 Duplicates (quick preview, details in chapter 08)

In [36]:
dup = pd.concat([df.head(3), df.head(2)], ignore_index=True)
print("rows:", len(dup), "| duplicates:", dup.duplicated().sum())

print(len(dup.drop_duplicates()))

rows: 5 | duplicates: 2
3


## 7.4 Dropping missing values (preview, details in chapter 07)

In [37]:
m = pd.DataFrame({"a": [1, np.nan, 3], "b": [4, 5, np.nan]})
print(m.dropna())                    # drop rows with any NaN
print(m.dropna(axis=1))              # drop columns with any NaN

     a    b
0  1.0  4.0
Empty DataFrame
Columns: []
Index: [0, 1, 2]


> **`inplace=True`**: older tutorials write `df.drop(..., inplace=True)`. It is **discouraged** now and does not save memory or time. Write `df = df.drop(...)`.

---
# 8. Adding rows

Tables grow mostly by **columns**. Adding rows one by one in a loop is slow. The right way: **collect new rows, then `concat` once**.

## 8.1 `pd.concat()`

In [38]:
new_rows = pd.DataFrame({
    "name": ["Isha", "Rohan"],
    "city": ["Pune", "Delhi"],
    "hours": [5.0, 3.5],
    "attendance": [75, 72],
    "prev_score": [62, 58],
    "score": [70, 52],
    "internet": ["Yes", "No"],
})

combined = pd.concat([df, new_rows], ignore_index=True)     # ignore_index gives a fresh 0..n-1 index
print(combined.shape)
combined.tail(3)

(10, 7)


,name,city,hours,attendance,prev_score,score,internet
7,Pooja,Indore,6.0,88,68,81,Yes
8,Isha,Pune,5.0,75,62,70,Yes
9,Rohan,Delhi,3.5,72,58,52,No


## 8.2 `loc` with a new label (fine for one row)

In [39]:
d = df.copy()
d.loc[len(d)] = ["Tara", "Pune", 4.5, 82, 66, 77, "Yes"]    # values in column order
d.tail(2)

,name,city,hours,attendance,prev_score,score,internet
7,Pooja,Indore,6.0,88,68,81,Yes
8,Tara,Pune,4.5,82,66,77,Yes


- Old `df.append()` was **removed** in pandas 2.0. Use `concat`.
- Never do `concat` inside a big loop. Put rows in a list, build one DataFrame at the end.

In [40]:
# The efficient pattern
rows = []
for i in range(3):
    rows.append({"name": f"Student{i}", "score": 50 + i * 10})

pd.DataFrame(rows)

,name,score
0,Student0,50
1,Student1,60
2,Student2,70


---
# 9. Moving and reordering columns

In [41]:
# Choose the order by selecting
df[["name", "score", "city", "hours", "attendance", "prev_score", "internet"]].head(2)

,name,score,city,hours,attendance,prev_score,internet
0,Ravi,85,Indore,5.5,80,70,Yes
1,Asha,92,Bhopal,7.0,90,78,Yes


In [42]:
# Move the target to the last column (common in ML datasets)
cols = [c for c in df.columns if c != "score"] + ["score"]
df[cols].head(2)

,name,city,hours,attendance,prev_score,internet,score
0,Ravi,Indore,5.5,80,70,Yes,85
1,Asha,Bhopal,7.0,90,78,Yes,92


In [43]:
# Sort column names alphabetically
df[sorted(df.columns)].head(2)

,attendance,city,hours,internet,name,prev_score,score
0,80,Indore,5.5,Yes,Ravi,70,85
1,90,Bhopal,7.0,Yes,Asha,78,92


---
# 10. Method chaining: `assign` and `pipe`

Instead of many lines with the same variable, write one readable **pipeline**: each step takes the result of the previous one.

In [44]:
# Step-by-step style
a = df.copy()
a["improvement"] = a["score"] - a["prev_score"]
a = a[a["improvement"] > 15]
a = a[["name", "city", "improvement"]]
a = a.sort_values("improvement", ascending=False)
a = a.reset_index(drop=True)
a

,name,city,improvement
0,Neha,Indore,16


In [45]:
# Same thing as one chain (no variable repeated)
result = (
    df
    .assign(improvement=lambda x: x["score"] - x["prev_score"])
    .query("improvement > 15")
    .loc[:, ["name", "city", "improvement"]]
    .sort_values("improvement", ascending=False)
    .reset_index(drop=True)
)
result

,name,city,improvement
0,Neha,Indore,16


In [46]:
# pipe(): plug your own function into a chain
def add_grade(data):
    data = data.copy()
    data["grade"] = np.select([data["score"] >= 90, data["score"] >= 75], ["A", "B"], default="C")
    return data

df.pipe(add_grade)[["name", "score", "grade"]].head(4)

,name,score,grade
0,Ravi,85,B
1,Asha,92,A
2,Karan,58,C
3,Neha,88,B


Chains make the **data cleaning steps** easy to read top to bottom, and each step is easy to test. This style is very common in professional notebooks.

---
# 11. Mini AI/ML example: feature engineering

Goal: prepare `df` so that a model can predict `score`.

Models need **numbers**, so we must:
1. create useful new features
2. convert text columns into numbers (**encoding**)
3. remove columns that are useless or leaky
4. produce the final `X` and `y`

In [47]:
# 1. New features
fe = df.assign(
    effort_index=lambda x: x["hours"] * x["attendance"] / 100,
    prev_hours_ratio=lambda x: x["prev_score"] / x["hours"],
    attendance_level=lambda x: pd.cut(x["attendance"], [0, 70, 85, 100], labels=["low", "medium", "high"]),
)
fe[["name", "effort_index", "prev_hours_ratio", "attendance_level"]]

,name,effort_index,prev_hours_ratio,attendance_level
0,Ravi,4.400,12.727273,medium
1,Asha,6.300,11.142857,high
2,Karan,1.950,18.333333,low
3,Neha,5.525,11.076923,medium
4,Amit,2.800,15.000000,low
5,Sara,7.600,10.625000,high
6,Dev,1.500,20.000000,low
7,Pooja,5.280,11.333333,high


In [48]:
# 2a. Binary text column -> 0/1 with map
fe["internet"] = fe["internet"].map({"Yes": 1, "No": 0})

# 2b. Text category with many values -> one-hot columns (one 0/1 column per category)
fe = pd.get_dummies(fe, columns=["city", "attendance_level"], dtype=int)
fe.head(3)

,name,hours,attendance,prev_score,score,internet,effort_index,prev_hours_ratio,city_Bhopal,city_Delhi,city_Indore,city_Pune,attendance_level_low,attendance_level_medium,attendance_level_high
0,Ravi,5.5,80,70,85,1,4.40,12.727273,0,0,1,0,0,1,0
1,Asha,7.0,90,78,92,1,6.30,11.142857,1,0,0,0,0,0,1
2,Karan,3.0,65,55,58,0,1.95,18.333333,0,1,0,0,1,0,0


In [49]:
# 3. Drop columns that are not features
fe = fe.drop(columns=["name"])          # a name has no predictive meaning

# 4. Final X and y
y = fe["score"]
X = fe.drop(columns=["score"])

print("X shape:", X.shape, "| y shape:", y.shape)
print(X.columns.tolist())

X shape: (8, 13) | y shape: (8,)
['hours', 'attendance', 'prev_score', 'internet', 'effort_index', 'prev_hours_ratio', 'city_Bhopal', 'city_Delhi', 'city_Indore', 'city_Pune', 'attendance_level_low', 'attendance_level_medium', 'attendance_level_high']


**Why `pd.get_dummies`?** A model cannot use "Indore". Giving `Indore=1, Bhopal=2` would wrongly suggest that Bhopal is "more" than Indore. One-hot columns avoid that false ordering.

In bigger projects, `sklearn.preprocessing.OneHotEncoder` is used so that exactly the **same columns** are created for the test data. `get_dummies` is perfect for learning and exploration.

---
# 12. Pandas 3.x notes

- **Copy-on-Write:** `d = df.copy()` is not even needed to protect `df` from selections, but we kept it in demos to be clear. Methods like `drop`, `rename`, `assign` always return new objects.
- Avoid `inplace=True`. Always `df = df.method(...)`.
- Chained assignment `df[cond]["col"] = x` does not work. Use `df.loc[cond, "col"] = x`.
- `pd.get_dummies` returns **boolean** columns by default. Use `dtype=int` when you need 0/1 numbers (as we did).
- Text columns use the `str` dtype; `.str` methods work the same.
- `DataFrame.append` does not exist. Use `pd.concat`.

---
# 13. Common mistakes

| Mistake | Fix |
|---|---|
| `df.drop("col")` and expecting `df` to change | `df = df.drop(columns="col")` |
| `df.drop("col")` (axis unclear, tries to drop a row) | `df.drop(columns="col")` |
| `df["new"] = [1, 2, 3]` with wrong length | Length must equal number of rows |
| Assigning a Series with a different index and getting NaN | Check/align the index, or use `.to_numpy()` when you mean position |
| Adding rows inside a loop with `concat` | Collect rows in a list, build once |
| `df.append(...)` | Removed. Use `pd.concat` |
| `df[cond]["col"] = value` | `df.loc[cond, "col"] = value` |
| `astype(int)` on a column with NaN | Fill NaN first or use `"Int64"` |
| Numbers stored as text stay text | `pd.to_numeric(..., errors="coerce")` |
| Using `apply(axis=1)` for simple maths | Use column operations or `np.where` |
| Label-encoding cities as 1, 2, 3 for ML | Use one-hot (`get_dummies`) |
| Keeping ID/name columns as features | Drop them |
| Keeping a column derived from the target as a feature | Data leakage: drop it |

---
# 14. Interview questions

**Q1. How do you add a new column based on a condition?**
`np.where(cond, a, b)` for two outcomes, `np.select(conds, choices, default)` for many outcomes, or `df.loc[cond, "col"] = value`.

**Q2. `drop` vs `pop` vs `del`?**
`drop` returns a new DataFrame and works for rows or columns. `pop` removes a column in place and returns it. `del df["col"]` removes a column in place and returns nothing.

**Q3. Why is `inplace=True` discouraged?**
It does not really save memory or time, makes chaining impossible, and behaves inconsistently. Assigning the returned object is clearer.

**Q4. How do you add rows to a DataFrame?**
`pd.concat([df, new_rows], ignore_index=True)`. `append` was removed in pandas 2. For many rows, collect them in a list and build the DataFrame once.

**Q5. `pd.cut` vs `pd.qcut`?**
`cut` uses bin edges you define. `qcut` makes bins with (nearly) equal number of rows using quantiles.

**Q6. `apply` vs vectorized operations?**
Vectorized operations run in compiled code on the entire column and are much faster. `apply` calls a Python function per element/row, so use it only when there is no vectorized alternative.

**Q7. How do you convert a column with bad text values into numbers?**
`pd.to_numeric(col, errors="coerce")`. Invalid values become NaN.

**Q8. What is one-hot encoding and how is it done in pandas?**
Each category becomes its own 0/1 column, so models do not assume a fake order. `pd.get_dummies(df, columns=[...], dtype=int)`.

**Q9. What does `assign` do and why is it useful?**
It returns a new DataFrame with extra or modified columns, so it can be used inside a method chain. Lambdas can refer to columns created earlier in the same call.

**Q10. What happens if you assign a Series to a column and the indexes differ?**
Pandas aligns by index label. Rows with no matching label get NaN.

---
# 15. Practice questions

Run this cell first to get the practice data.

In [50]:
orders = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 5, 6, 7, 8],
    "customer":   ["Ravi", "Asha", "Ravi", "Neha", "Amit", "Asha", "Sara", "Ravi"],
    "product":    ["Pen", "Book", "Bag", "Pen", "Book", "Pen", "Bag", "Book"],
    "price":      [10, 250, 800, 10, 250, 10, 800, 250],
    "qty":        [10, 2, 1, 25, 3, 5, 2, 1],
    "discount":   [0, 10, 0, 5, 0, 0, 15, 10],
    "order_date": ["2026-01-05", "2026-01-12", "2026-02-03", "2026-02-18",
                   "2026-03-07", "2026-03-21", "2026-04-02", "2026-04-15"],
})
orders

,order_id,customer,product,price,qty,discount,order_date
0,1,Ravi,Pen,10,10,0,2026-01-05
1,2,Asha,Book,250,2,10,2026-01-12
2,3,Ravi,Bag,800,1,0,2026-02-03
3,4,Neha,Pen,10,25,5,2026-02-18
4,5,Amit,Book,250,3,0,2026-03-07
5,6,Asha,Pen,10,5,0,2026-03-21
6,7,Sara,Bag,800,2,15,2026-04-02
7,8,Ravi,Book,250,1,10,2026-04-15


**Q1.** Add a column `gross` = `price * qty`.
**Q2.** Add `net` = `gross` after discount (discount is in percent), rounded to 2 decimals.
**Q3.** Add `size`: `"bulk"` if `qty >= 10`, else `"normal"`.
**Q4.** Add `tier` using `pd.cut` on `net` with bins `[0, 100, 500, 5000]` and labels `low`, `mid`, `high`.
**Q5.** Convert `order_date` to datetime and add a `month` column with the month name.
**Q6.** Rename all columns to upper case.
**Q7.** Drop `discount` and `gross` columns.
**Q8.** Add a new order (order_id 9, Isha, Pen, price 10, qty 4, discount 0, date 2026-04-20) using `concat`.
**Q9.** Insert a column `currency` with value `"INR"` at position 1.
**Q10.** One-hot encode `product` (as 0/1 integers) and attach it to the table.
**Q11.** In **one method chain**: add `net`, keep only orders with `net > 100`, keep columns `customer`, `product`, `net`, and sort by `net` descending.
**Q12.** Add `customer_total`: the total `net` of each customer, repeated on every row of that customer (use `groupby().transform`).

### Solutions

In [51]:
# Q1
o = orders.copy()
o["gross"] = o["price"] * o["qty"]
o[["order_id", "price", "qty", "gross"]].head(3)

,order_id,price,qty,gross
0,1,10,10,100
1,2,250,2,500
2,3,800,1,800


In [52]:
# Q2
o = orders.assign(gross=lambda x: x["price"] * x["qty"])
o["net"] = (o["gross"] * (1 - o["discount"] / 100)).round(2)
o[["order_id", "gross", "discount", "net"]]

,order_id,gross,discount,net
0,1,100,0,100.0
1,2,500,10,450.0
2,3,800,0,800.0
3,4,250,5,237.5
4,5,750,0,750.0
5,6,50,0,50.0
6,7,1600,15,1360.0
7,8,250,10,225.0


In [53]:
# Q3
o = orders.copy()
o["size"] = np.where(o["qty"] >= 10, "bulk", "normal")
o[["order_id", "qty", "size"]]

,order_id,qty,size
0,1,10,bulk
1,2,2,normal
2,3,1,normal
3,4,25,bulk
4,5,3,normal
5,6,5,normal
6,7,2,normal
7,8,1,normal


In [54]:
# Q4
o = orders.assign(net=lambda x: (x["price"] * x["qty"] * (1 - x["discount"] / 100)).round(2))
o["tier"] = pd.cut(o["net"], bins=[0, 100, 500, 5000], labels=["low", "mid", "high"])
o[["order_id", "net", "tier"]]

,order_id,net,tier
0,1,100.0,low
1,2,450.0,mid
2,3,800.0,high
3,4,237.5,mid
4,5,750.0,high
5,6,50.0,low
6,7,1360.0,high
7,8,225.0,mid


In [55]:
# Q5
o = orders.copy()
o["order_date"] = pd.to_datetime(o["order_date"])
o["month"] = o["order_date"].dt.month_name()
o[["order_id", "order_date", "month"]].head(4)

,order_id,order_date,month
0,1,2026-01-05,January
1,2,2026-01-12,January
2,3,2026-02-03,February
3,4,2026-02-18,February


In [56]:
# Q6
orders.rename(columns=str.upper).columns.tolist()

['ORDER_ID', 'CUSTOMER', 'PRODUCT', 'PRICE', 'QTY', 'DISCOUNT', 'ORDER_DATE']

In [57]:
# Q7
o = orders.assign(gross=lambda x: x["price"] * x["qty"])
o = o.drop(columns=["discount", "gross"])
o.columns.tolist()

['order_id', 'customer', 'product', 'price', 'qty', 'order_date']

In [58]:
# Q8
new = pd.DataFrame([{"order_id": 9, "customer": "Isha", "product": "Pen", "price": 10,
                     "qty": 4, "discount": 0, "order_date": "2026-04-20"}])
o = pd.concat([orders, new], ignore_index=True)
print(o.shape)
o.tail(2)

(9, 7)


,order_id,customer,product,price,qty,discount,order_date
7,8,Ravi,Book,250,1,10,2026-04-15
8,9,Isha,Pen,10,4,0,2026-04-20


In [59]:
# Q9
o = orders.copy()
o.insert(1, "currency", "INR")
o.head(3)

,order_id,currency,customer,product,price,qty,discount,order_date
0,1,INR,Ravi,Pen,10,10,0,2026-01-05
1,2,INR,Asha,Book,250,2,10,2026-01-12
2,3,INR,Ravi,Bag,800,1,0,2026-02-03


In [60]:
# Q10
o = pd.get_dummies(orders, columns=["product"], dtype=int)
o.head(3)

,order_id,customer,price,qty,discount,order_date,product_Bag,product_Book,product_Pen
0,1,Ravi,10,10,0,2026-01-05,0,0,1
1,2,Asha,250,2,10,2026-01-12,0,1,0
2,3,Ravi,800,1,0,2026-02-03,1,0,0


In [61]:
# Q11
(
    orders
    .assign(net=lambda x: (x["price"] * x["qty"] * (1 - x["discount"] / 100)).round(2))
    .query("net > 100")
    .loc[:, ["customer", "product", "net"]]
    .sort_values("net", ascending=False)
)

,customer,product,net
6,Sara,Bag,1360.0
2,Ravi,Bag,800.0
4,Amit,Book,750.0
1,Asha,Book,450.0
3,Neha,Pen,237.5
7,Ravi,Book,225.0


In [62]:
# Q12
o = orders.assign(net=lambda x: (x["price"] * x["qty"] * (1 - x["discount"] / 100)).round(2))
o["customer_total"] = o.groupby("customer")["net"].transform("sum")
o[["order_id", "customer", "net", "customer_total"]]

,order_id,customer,net,customer_total
0,1,Ravi,100.0,1125.0
1,2,Asha,450.0,500.0
2,3,Ravi,800.0,1125.0
3,4,Neha,237.5,237.5
4,5,Amit,750.0,750.0
5,6,Asha,50.0,500.0
6,7,Sara,1360.0,1360.0
7,8,Ravi,225.0,1125.0


---
# 16. Cheat sheet

```python
# ADD columns
df["new"] = 0                                   # constant
df["new"] = df["a"] * df["b"]                   # from columns
df["new"] = np.where(cond, "x", "y")            # 2 outcomes
df["new"] = np.select(conds, choices, default="z")
df["bin"] = pd.cut(df["a"], bins=[...], labels=[...])
df["q"]   = pd.qcut(df["a"], q=4)
df["new"] = df["col"].map({"Yes": 1, "No": 0})
df["avg"] = df.groupby("g")["a"].transform("mean")
df = df.assign(new=lambda x: x["a"] + 1)
df.insert(1, "new", values)

# UPDATE values
df.loc[cond, "col"] = value
df["col"] = df["col"].replace({"old": "new"})
df["col"] = df["col"].clip(lower=0, upper=100)
df["col"] = df["col"].apply(func)               # last resort

# TYPES
df["a"].astype("float64")   pd.to_numeric(s, errors="coerce")
pd.to_datetime(s)           s.astype("category")

# RENAME
df.rename(columns={"old": "new"})
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# DROP
df = df.drop(columns=["a", "b"])    df = df.drop(index=[0, 1])
df.pop("a")                         del df["a"]
df = df.drop_duplicates()           df = df.dropna()

# ADD rows
df = pd.concat([df, new_rows], ignore_index=True)

# ENCODE for ML
pd.get_dummies(df, columns=["city"], dtype=int)

# CHAIN
(df.assign(...).query("...").loc[:, [...]].sort_values("...").reset_index(drop=True))
```

**Next chapter:** `07_handling_missing_data`: why data goes missing, `isna`, `dropna`, `fillna` (mean, median, mode, forward fill, group-wise), interpolation, and how to decide which method to use.